Notebook 1: Configuração do ambiente e carregamento do dataset


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

arquivo_nome = "refrigeradores-e-assemelhados.csv"

diretorio_atual = Path.cwd()

# Busca dinâmica pelo arquivo no diretório atual e superiores
caminhos_encontrados = list(diretorio_atual.rglob(arquivo_nome))

if not caminhos_encontrados:
    caminhos_encontrados = list(diretorio_atual.parent.rglob(arquivo_nome))

if caminhos_encontrados:
    caminho_final = caminhos_encontrados[0]
    print(f"Arquivo localizado com sucesso em: {caminho_final}")
    
    df = pd.read_csv(caminho_final)

    # Padronização de nomes das colunas
    df.columns = (
        df.columns.str.strip()
        .str.replace(' ', '_')
        .str.replace('(', '')
        .str.replace(')', '')
        .str.replace('°C', 'C')
        .str.replace('/', '_')
        .str.replace('%', 'pct')
    )
    
    print("Dataset carregado e colunas padronizadas!")
else:
    caminho_manual = Path("../Datasets Itens de casa/refrigeradores-e-assemelhados.csv")
    df = pd.read_csv(caminho_manual)
    df.columns = (
        df.columns.str.strip()
        .str.replace(' ', '_')
        .str.replace('(', '')
        .str.replace(')', '')
        .str.replace('°C', 'C')
        .str.replace('/', '_')
        .str.replace('%', 'pct')
    )
    print("Carregado via caminho manual!")

Arquivo localizado com sucesso em: c:\Users\gianl\OneDrive\Área de Trabalho\ProjetoTCS\Datasets\refrigeradores-e-assemelhados.csv
Dataset carregado e colunas padronizadas!


Notebook 2: Seleção de atributos (Feature selection)

In [4]:
colunas_para_remover = [
    'Se_outro_fluido_refrigerante,_especificar',
    'Se_outro_agente_de_expansão,_especificar',
    'Código_de_Barras',
    'Status_do_Modelo',
    'Status_do_Registro',
    'Número_do_Registro',
    'Norma_de_Segurança_Aplicada_IEC_60335-2-24'
]

colunas_existentes = [col for col in colunas_para_remover if col in df.columns]
df = df.drop(columns=colunas_existentes)

print(f"✓ {len(colunas_existentes)} colunas irrelevantes removidas.")
print(f"Colunas restantes ({len(df.columns)}):", list(df.columns))

✓ 7 colunas irrelevantes removidas.
Colunas restantes (24): ['Marca', 'Modelo', 'Tipo', 'Tensão_V', 'Sistema_de_Degelo', 'Agente_de_Expansão_da_Espuma', 'Fluido_refrigerante', 'Massa_do_fluido_refrigerante_g', 'Capacidade_de_Congelamento_kg_24_h', 'Retenção_de_Temperatura_h', 'Classificação_do_Compartimento_Congelador_ou_de_Alimentos_Congelados', 'Volume_do_Refrigerador_litros', 'Volume_do_Compartimento_de_Alimentos_Congelados_do_Tipo_1_Estrela_litros', 'Volume_do_Compartimento_de_Alimentos_Congelados_do_Tipo_2_Estrelas_litros', 'Volume_do_Compartimento_de_Alimentos_Congelados_do_Tipo_3_Estrelas_litros', 'Volume_Total_litros', 'Consumo_de_Energia_Mensal_kWh_mês', 'Índice_de_Eficiência_Padrão_2022_pct', 'Classe_de_Eficiência_Energética_Padrão_2022', 'Consumo_de_Energia_Diário_a_16_C_kWh_dia', 'Consumo_de_Energia_Diário_a_32_C_kWh_dia', 'Consumo_Médio_de_Energia_Mensal_kWh_mês', 'Índice_de_Eficiência_Padrão_2025_pct', 'Classe_de_Eficiência_Energética_Padrão_2025']


Notebook 3: Tratamento de Duplicatas e Registros Redundantes

In [5]:
linhas_antes = len(df)
df = df.drop_duplicates()
print(f"✓ Linhas duplicadas exatas removidas: {linhas_antes - len(df)}")

if 'Modelo' in df.columns:
    # Prioriza o registro que tiver o menor número de valores nulos entre as duplicatas do mesmo modelo
    df['contagem_nulos'] = df.isnull().sum(axis=1)
    df = df.sort_values('contagem_nulos').drop_duplicates(subset=['Modelo'], keep='first')
    df = df.drop(columns=['contagem_nulos'])
    print(f"✓ Modelos duplicados tratados. Total de linhas atual: {len(df)}")

✓ Linhas duplicadas exatas removidas: 121
✓ Modelos duplicados tratados. Total de linhas atual: 2642


Notebook 4: Filtragem de Erros e Outliers Irrealistas

In [6]:
# Unificação/Ajuste das colunas de consumo de energia
coluna_consumo = 'Consumo_de_Energia_Mensal_kWh_mês'
if coluna_consumo not in df.columns and 'Consumo_Médio_de_Energia_Mensal_kWh_mês' in df.columns:
    coluna_consumo = 'Consumo_Médio_de_Energia_Mensal_kWh_mês'

# Filtragem de valores inconsistentes ou zerados de consumo energético (mínimo de 5 kWh/mês para refrigeradores)
if coluna_consumo in df.columns:
    df = df[df[coluna_consumo] >= 5.0]

# Filtro de volume total irrealista (elimina registros sem volume ou abaixo de 10 litros)
if 'Volume_Total_litros' in df.columns:
    df = df[df['Volume_Total_litros'] >= 10.0]

print(f"✓ Filtros de inconsistência aplicados. Linhas restantes: {len(df)}")

✓ Filtros de inconsistência aplicados. Linhas restantes: 1075


Notebook 5: Imputação Inteligente e Exportação dos Dados Limpos

In [7]:
colunas_imputar = [
    coluna_consumo,
    'Volume_Total_litros'
]

colunas_validas = [c for c in colunas_imputar if c in df.columns]

# Imputação pela média do agrupamento por Marca e Tipo/Classe energética
agrupadores = [c for c in ['Marca', 'Tipo', 'Classe_de_Eficiência_Energética_Padrão_2022'] if c in df.columns]

if agrupadores and colunas_validas:
    df[colunas_validas] = df.groupby(agrupadores)[colunas_validas].transform(lambda x: x.fillna(x.mean()))

# Caso ainda restem nulos em marcas sem outros modelos para cálculo da média, preenche pela média do Tipo
if 'Tipo' in df.columns and colunas_validas:
    df[colunas_validas] = df.groupby('Tipo')[colunas_validas].transform(lambda x: x.fillna(x.mean()))

# Remove registros que ainda permanecerem sem métrica básica de consumo
df = df.dropna(subset=colunas_validas)

print(f"✓ Processo finalizado! Linhas prontas para uso: {len(df)}")

# Salva o dataset limpo no mesmo diretório do arquivo original
caminho_saida = caminho_final.parent / "refrigeradores-e-assemelhados_limpo.csv"
df.to_csv(caminho_saida, index=False)
print(f"✓ Dataset limpo salvo com sucesso em: {caminho_saida}")

✓ Processo finalizado! Linhas prontas para uso: 1075
✓ Dataset limpo salvo com sucesso em: c:\Users\gianl\OneDrive\Área de Trabalho\ProjetoTCS\Datasets\refrigeradores-e-assemelhados_limpo.csv
